In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import binned_statistic
from astropy import constants as c

In [ ]:
from specs import *
from classy_wraper_for_m21cm import *

In [ ]:
from meer21cm import util
from meer21cm.plot import plot_map
from meer21cm import PowerSpectrum
from meer21cm.power import get_modelpk_conv

In [ ]:
ps = PowerSpectrum(
    wproj=wcs,
    num_pix_x=num_pix_x,
    num_pix_y=num_pix_y,
    nu=nu_arr,
    ra_range=ra_range,
    dec_range=dec_range,
    # downscale the resolution along line-of-sight
    downres_factor_radial = 1.5,
    # downscale the resolution on the transverse plane
    downres_factor_transverse = 1.2,
    omega_hi = 5e-4,
    tracer_bias_1 = 1.5,
    mean_amp_1 = 'average_hi_temp'
)

In [ ]:
ps.get_enclosing_box()
plot_map(ps.W_HI,ps.wproj,have_cbar=False)
ps.box_len

In [ ]:
print("total area of the survey is %.1f sqdeg" % (ps.W_HI[:,:,0].sum() * ps.pixel_area))
print("redshift range is %.1f - %.1f" % (ps.z_ch[-1], ps.z_ch[0]))

In [ ]:
# |k| and \mu=k_para/|k|, with the 3D shape of the box
ps.kmode.shape,ps.mumode.shape, ps.k_nyquist

In [ ]:
power_model_3d = ps.auto_power_matter_model
power_model_3d.shape

In [ ]:
kmode = ps.kmode

kmask = kmode > 0.0
Nke = 20

kclean = kmode[kmask]
muclean = ps.mumode[kmask]

kedges = np.geomspace(
    np.min(kclean), np.max(kclean), Nke
)
Nmodes, _, _ = binned_statistic(kclean, [], "count", kedges)
Nmask = ~(Nmodes==0.0)
k = np.sqrt(kedges[1:] * kedges[:-1])[Nmask]
k

In [ ]:
Nmodes

## Lets try an a bit more involved PS 

In [ ]:
ps.cosmo # This seems to be an astropy object

In [ ]:
print(ps.fiducial_cosmology)
print(ps.true_cosmology)

In [ ]:
cosmo = Class_cosmo_model(ps.true_cosmology)

In [ ]:
Pk_nw = cosmo.Pk_nw(k, 1)
plt.loglog(k, Pk_nw)
plt.loglog(k, cosmo.Pk_lin(k, 1), ls="--")

In [ ]:
z = 0
mu = np.linspace(-1, 1, 9)

sigmav = cosmo.sigmav(z)
print(sigmav)

Pk_nw = cosmo.Pk_nw(k, z)
Pk_QNL = cosmo.Pk_QNL(k[:, None], mu[None, :], z, sigmav)

Pk_QNL_noBB = np.squeeze(Pk_QNL / Pk_nw[:, None, :] - 1)
plt.semilogx(k, Pk_QNL_noBB)

## $ k $ sampling avergeraging
Vectorise the functions

In [ ]:
sigmav = cosmo.sigmav(ps.z)
Pqnl_samples = cosmo.Pk_QNL(ps.kmode, ps.mumode, ps.z, sigmav)

## Testing AP

In [ ]:
true_cosmology = {"h":0.71, "omega_baryon":0.05, "omega_cold":0.35}
print(*cosmo.transform_input_dict_to_class(true_cosmology))
# ps.true_cosmology = true_cosmology
# ps.alpha_parallel, ps.alpha_perp

In [ ]:
#class instance for new fid cosmology
cosmo = Class_cosmo_model(ps.fiducial_cosmology)
# cosmo_true = Class_cosmo_model(true_cosmology)
cosmo_true = cosmo

In [ ]:
def alpha_parallel(true:Class_cosmo_model, fiducial:Class_cosmo_model, z):
    Hr = true.Hubble(z) * true.rsdrag
    Hr_fid = fiducial.Hubble(z) * fiducial.rsdrag
    return Hr_fid / Hr

def alpha_perp(true:Class_cosmo_model, fiducial:Class_cosmo_model, z):
    DAr = true.comoving_Distance(z) / true.rsdrag
    DAr_fid = fiducial.comoving_Distance(z) / fiducial.rsdrag
    return DAr / DAr_fid

In [ ]:
alpha_parallel(cosmo_true, cosmo, ps.z), alpha_perp(cosmo_true, cosmo, ps.z)

In [ ]:
z = np.linspace(0, 5)
chi = cosmo_true.comoving_Distance(z) # Mpc 

chi_21cm = ps.astropy_cosmo_true.comoving_distance(z)
plt.semilogy(z, chi)
plt.semilogy(z, chi_21cm, ls="--")

plt.xlabel("$z$")
plt.ylabel("$\chi\,[\mathrm{Mpc}]$")

In [ ]:
z = np.linspace(0, 5)
H = cosmo_true.Hubble(z) # Mpc 

H_21cm = ps.astropy_cosmo_true.H(z)
plt.semilogy(z, H * c.c / 1e3)
plt.semilogy(z, H_21cm, ls="--")

plt.xlabel("$z$")
plt.ylabel("$H(z)\,[\mathrm{km}\,\mathrm{s}^{-1}\,\mathrm{Mpc}^{-1}]$")

In [ ]:
cosmo.rsdrag, ps.sound_horizon_drag_fiducial

In [ ]:
cosmo_true.rsdrag, ps.sound_horizon_drag_true

## RSD

In [ ]:
print(ps.tracer_bias_1, "hi bias")
print(ps.sigma_v_1, "FOG size")

In [ ]:
T2 = ps.average_hi_temp**2
kaiser = (ps.tracer_bias_1 + cosmo_true.f_lin(ps.z) * muclean**2)**2

sigmaV = cosmo.sigmav(ps.z)
pqnl = cosmo.Pk_QNL(kclean, muclean, ps.z, sigmaV).squeeze(-1)
pmm = cosmo.Pk_lin(kclean, ps.z).squeeze(-1)

Ps_rsd_noobs_class = T2 * kaiser * pqnl
Ps_rsd_noqnl_noobs_class = T2 * kaiser * pmm
Ps_norsd_noobs_class = T2 * ps.tracer_bias_1**2 * pqnl
Ps_norsd_noqnl_noobs_class = T2 * ps.tracer_bias_1**2 * pmm

Do some mild averaging

In [ ]:
Ps_1d_rsd_noobs_class, _, _ = binned_statistic( kclean, Ps_rsd_noobs_class, "mean", kedges)
Ps_1d_rsd_noqnl_noobs_class, _, _ = binned_statistic( kclean, Ps_rsd_noqnl_noobs_class, "mean", kedges)
Ps_1d_norsd_noobs_class, _, _ = binned_statistic( kclean, Ps_norsd_noobs_class, "mean", kedges)
Ps_1d_norsd_noqnl_noobs_class, _, _ = binned_statistic( kclean, Ps_norsd_noqnl_noobs_class, "mean", kedges)

In [ ]:
plt.loglog(k, Ps_1d_rsd_noqnl_noobs_class[Nmask], label="Linear matter only")
plt.loglog(k, Ps_1d_rsd_noobs_class[Nmask], ls="--", label="Dewiggled")

plt.xlabel(r"Wavenumber $k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"1D Powerspectrum $P\,[\mathrm{Mpc}^{3}]$")
plt.legend()
plt.grid()

## Beam and pixelisation

In [ ]:
ps.sigma_beam_ch = 0.4
beam2 = ps.beam_attenuation()[kmask]**2

In [ ]:
ps.grid_scheme
ps.grid_scheme = "cic"
gridwin = ps.gridding_compensation()[kmask]**2

In [ ]:
print(ps.sampling_resol)
ps.sampling_resol = [
    ps.pix_resol_in_mpc,
    ps.pix_resol_in_mpc,
    ps.los_resol_in_mpc, # Is this in fid or true
]
pixwin = ps.map_sampling()[kmask]**2 

In [ ]:
ps.sampling_resol

In [ ]:
ps.z

In [ ]:
Ps_rsd_obs_class = Ps_rsd_noobs_class * beam2 * gridwin * pixwin
Ps_rsd_noqnl_obs_class = Ps_rsd_noqnl_noobs_class * beam2 * gridwin * pixwin

In [ ]:
Ps_1d_rsd_obs_class, _, _ = binned_statistic(kclean, Ps_rsd_obs_class, "mean", kedges)
Ps_1d_rsd_noqnl_obs_class, _, _ = binned_statistic(kclean, Ps_rsd_noqnl_obs_class, "mean", kedges)

In [ ]:
plt.loglog(k, Ps_1d_rsd_noqnl_obs_class[Nmask], label="Linear matter only")
plt.loglog(k, Ps_1d_rsd_obs_class[Nmask], label="Dewiggled", ls="--")
plt.loglog(k, Ps_1d_rsd_noobs_class[Nmask], label="No observational effects")

plt.xlabel(r"Wavenumber $k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"1D Powerspectrum $P\,[\mathrm{Mpc}^{3}]$")
plt.legend()
plt.grid()

## Konvolution with survey weight function

In [ ]:
ps.grid_data_to_field()

fig, axes = plt.subplots(1,3,figsize=(15,5))
for i, ax in enumerate(axes):
    ax.imshow(ps.counts_in_box.mean(i),aspect='auto')

In [ ]:
ps.weights_1 = ps.counts_in_box.astype('float')

In [ ]:
def restore_shape(pk):
    normal = np.zeros_like(kmode)
    normal[kmask] = pk
    return normal

In [ ]:
Ps_survey_model = get_modelpk_conv(
    restore_shape(Ps_rsd_obs_class),
    ps.weights_1,
    ps.weights_1,
)[kmask]

Ps_survey_model_noqnl = get_modelpk_conv(
    restore_shape(Ps_rsd_noqnl_obs_class),
    ps.weights_1,
    ps.weights_1,
)[kmask]

In [ ]:
Ps_survey_model_1d, _, _ = binned_statistic(kclean, Ps_survey_model, "mean", kedges)
Ps_survey_model_1d_noqnl, _, _ = binned_statistic(kclean, Ps_survey_model_noqnl, "mean", kedges)

In [ ]:
plt.loglog(k, Ps_survey_model_1d[Nmask])
plt.loglog(k, Ps_survey_model_1d_noqnl[Nmask], ls="--")

# Do some fake(!) "broadband extraction"

In [ ]:
Pnw = cosmo.Pk_nw(kclean, ps.z).squeeze(-1)
Pnw_noobs = T2 * kaiser * Pnw

Ps_1d_nw_noobs, _, _ = binned_statistic(kclean, Pnw_noobs, "mean", kedges)
Ps_1d_nw_noobs = Ps_1d_nw_noobs[Nmask]

Pnw_obs = Pnw_noobs * beam2 * gridwin * pixwin

Pnw_obs_konv = get_modelpk_conv(
    restore_shape(Pnw_obs),
    ps.weights_1,
    ps.weights_1,
)[kmask]

Ps_1d_nw, _, _ = binned_statistic(kclean, Pnw_obs_konv, "mean", kedges)
Ps_1d_nw = Ps_1d_nw[Nmask]

In [ ]:
plt.semilogx(k, Ps_1d_rsd_noobs_class / Ps_1d_nw_noobs - 1, label="No Obs. Effects")
plt.semilogx(k, Ps_survey_model_1d / Ps_1d_nw - 1, label="Obs. Effects")

plt.xlabel(r"Wavenumber $k\,[\mathrm{Mpc}^{-1}]$")
plt.ylabel(r"Extracted BAO wiggle")
plt.legend()
plt.grid()

# Compare to meer21cm

In [ ]:
ps.k1dbins = kedges

In [ ]:
Pk_1d_noobs_norsd_matter_meer21cm, keff, _ = ps.get_1d_power(ps.auto_power_matter_model_r)
Pk_1d_noobs_norsd_matter_class, _, _ = binned_statistic(kclean, cosmo.Pk_lin(kclean, ps.z).squeeze(-1), "mean", kedges)

In [ ]:
plt.loglog(keff, Pk_1d_noobs_norsd_matter_meer21cm)
plt.loglog(k, Pk_1d_noobs_norsd_matter_class[Nmask])

In [ ]:
Pk_1d_noobs_rsd_meer21cm, keff, _ = ps.get_1d_power(
    ps.auto_power_tracer_1_model_noobs * ps.average_hi_temp**2
)

Pk_1d_noobs_rsd_class, _, _ = binned_statistic(
    kclean,
    (
        ps.average_hi_temp**2
        * (ps.tracer_bias_1 + cosmo.f_lin(ps.z) * muclean**2)**2
        * cosmo.Pk_lin(kclean, ps.z).squeeze(-1)
    ).flatten(),
    "mean",
    kedges
)

In [ ]:
plt.loglog(keff, Pk_1d_noobs_rsd_meer21cm)
plt.loglog(k, Pk_1d_noobs_rsd_class[Nmask])

In [ ]:
ps.include_beam = [True, False]
ps.include_sky_sampling = [True, False]
ps.compensate = [True, True]

In [ ]:
Pk_meer21cm = ps.auto_power_tracer_1_model
Pk_1d_meer21cm, k21cm, _ = ps.get_1d_power(
    Pk_meer21cm
)

In [ ]:
plt.loglog(k21cm, Pk_1d_meer21cm)
plt.loglog(k, Ps_survey_model_1d[Nmask])

# Looks fine